# CAMELS — U-Net Baseline (slim)

Segmentazione 2D aloni di materia oscura. Input: patch da `CAMELS_data\input_data\` (prodotte da `make_input_patches.ipynb`).

- U-Net minimal encoder-decoder con skip connections (~1.75M parametri), input `(1,256,256)` → mappa di probabilita` `(1,256,256)`.
- Loss: **BCE + Dice** (le patch negative sono tenute di proposito; la Dice stabilizza lo sbilanciamento).
- Metrica: **IoU/Dice** a soglia 0.5.
- Nota metodologica (prof., 29/09): una rete addestrata su proiezioni 2D **non puo` battere FoF+unbinding** (che usa l'informazione 3D completa); il valore e` velocita` + completezza. Lo stadio 2 (dalla maschera al catalogo: connectivity / CNN locale) e` un notebook separato.

In [ ]:
# ============ CONFIG ============
import os
DATA_DIR   = r"C:\Users\kekko\Downloads\Python\TESI\CAMELS_data"
INPUT_DIR  = os.path.join(DATA_DIR, "input_data")   # output di make_input_patches.ipynb
RUN_DIR    = os.path.join(DATA_DIR, "runs", "unet_baseline")

BATCH_SIZE   = 8
N_EPOCHS     = 100
LR           = 2e-4
WEIGHT_DECAY = 1e-4
VAL_FRACTION, TEST_FRACTION = 0.15, 0.15
SEED         = 42
THRESHOLD    = 0.5

# ATTENZIONE overfitting: 400 patch vengono da UNA sola simulazione e le patch adiacenti
# sono correlate. Piano: passare a 5-10 simulazioni CAMELS (2000-4000 patch); a quel punto
# probabilmente non serve data augmentation.


In [ ]:
import os, time, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED); np.random.seed(SEED)
print("device:", DEVICE)

X = np.load(os.path.join(INPUT_DIR, "X_patches.npy"))          # (N,1,256,256) f32
Y = np.load(os.path.join(INPUT_DIR, "Y_mask_patches.npy"))     # (N,1,256,256) {0,1}
print(f"X {X.shape} | Y {Y.shape} | px positivi: {Y.mean()*100:.2f}%")

perm = np.random.permutation(len(X))
n_val, n_test = int(len(X)*VAL_FRACTION), int(len(X)*TEST_FRACTION)
idx_tr, idx_va, idx_te = perm[n_val+n_test:], perm[:n_val], perm[n_val:n_val+n_test]

class HaloDataset(Dataset):
    def __init__(self, idx):
        self.x = torch.from_numpy(X[idx]); self.y = torch.from_numpy(Y[idx])
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i], self.y[i]

dl_tr = DataLoader(HaloDataset(idx_tr), BATCH_SIZE, shuffle=True,  drop_last=False)
dl_va = DataLoader(HaloDataset(idx_va), BATCH_SIZE, shuffle=False)
dl_te = DataLoader(HaloDataset(idx_te), BATCH_SIZE, shuffle=False)
print(f"train {len(idx_tr)} | val {len(idx_va)} | test {len(idx_te)}")


## U-Net minimal — 4 livelli, canali 64·2^l

In [ ]:
def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
    )

class UNet(nn.Module):
    """U-Net minimal: encoder a 4 livelli (base*2^l) + bottleneck; skip dal livello 0 al 2.
    downs[i] produce ch[i]; skips[i] ha ch[i] canali alla risoluzione img/2^i."""
    def __init__(self, c_in=1, c_out=1, base=64):
        super().__init__()
        ch = [base * 2**i for i in range(4)]            # 64,128,256,512
        enc_in = [c_in] + ch[:3]
        self.downs = nn.ModuleList([conv_block(enc_in[i], ch[i]) for i in range(4)])
        self.pools = nn.ModuleList([nn.MaxPool2d(2) for _ in range(3)])
        self.bottleneck = conv_block(ch[-1], ch[-1]*2)  # 512 -> 1024 (risoluzione img/8)
        # decoder: ups[j] riporta la risoluzione x2 e porta ai canali dello skip corrispondente
        self.ups = nn.ModuleList([
            nn.ConvTranspose2d(ch[3]*2, ch[2], 2, stride=2),   # 1024 -> 256
            nn.ConvTranspose2d(ch[2],   ch[1], 2, stride=2),   #  256 -> 128
            nn.ConvTranspose2d(ch[1],   ch[0], 2, stride=2),   #  128 ->  64
        ])
        self.decoding = nn.ModuleList([conv_block(2*c, c) for c in (ch[2], ch[1], ch[0])])
        self.head = nn.Conv2d(base, c_out, 1)
    def forward(self, x):
        skips = []
        for i in range(3):
            x = self.downs[i](x); skips.append(x); x = self.pools[i](x)
        x = self.downs[3](x)                 # ch[3] canali, img/8
        x = self.bottleneck(x)               # ch[3]*2 canali
        for up, dec, sk in zip(self.ups, self.decoding, skips[::-1]):
            x = torch.cat([up(x), sk], dim=1); x = dec(x)
        return self.head(x)                  # logits, stessa risoluzione dell'input

model = UNet().to(DEVICE)
n_par = sum(p.numel() for p in model.parameters())
print(f"parametri: {n_par:,}")
assert model(torch.zeros(1,1,256,256, device=DEVICE)).shape == (1,1,256,256)

## Loss e metriche

In [ ]:
def dice_loss(logits, target, eps=1.0):
    p = torch.sigmoid(logits).flatten(1); t = target.flatten(1)
    inter = (p * t).sum(1); denom = p.sum(1) + t.sum(1)
    return (1 - (2*inter + eps) / (denom + eps)).mean()

def criterion(logits, target):
    return F.binary_cross_entropy_with_logits(logits, target) + dice_loss(logits, target)

@torch.no_grad()
def evaluate(model, dl):
    model.eval()
    tot_iou, tot_n, losses = 0.0, 0, 0.0
    for xb, yb in dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        losses += float(criterion(logits, yb)) * len(xb)
        p = (torch.sigmoid(logits) >= THRESHOLD).float()
        inter = (p * yb).flatten(1).sum(1)
        union = ((p + yb) > 0).float().flatten(1).sum(1)
        iou = (inter + 1e-8) / (union + 1e-8)
        # patch senza label e senza predizioni = IoU 1 per definizione vuota: contiamo solo quelle informative
        informative = ~((yb.flatten(1).sum(1) == 0) & (p.flatten(1).sum(1) == 0))
        tot_iou += float(iou[informative].sum()); tot_n += int(informative.sum())
    model.train()
    return losses / len(dl.dataset), tot_iou / max(tot_n, 1)


## Training (checkpoint del best-val-IoU)

In [ ]:
os.makedirs(RUN_DIR, exist_ok=True)
opt = torch.optim.AdamW(model.parameters(), LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=N_EPOCHS)

best_iou, history = -1.0, []
t0 = time.time()
for epoch in range(N_EPOCHS):
    model.train(); run_loss = 0.0
    for xb, yb in dl_tr:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad(); loss = criterion(model(xb), yb)
        loss.backward(); opt.step()
    sched.step()
    val_loss, val_iou = evaluate(model, dl_va)
    run_loss /= len(dl_tr)
    history.append(dict(epoch=epoch, train_loss=run_loss, val_loss=val_loss, val_iou=val_iou))
    print(f"[{epoch:3d}] loss {run_loss:.4f} | val {val_loss:.4f} | val IoU {val_iou:.4f}")
    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), os.path.join(RUN_DIR, "best_unet.pt"))

with open(os.path.join(RUN_DIR, "history.json"), "w") as f:
    json.dump(history, f, indent=1)
print(f"tempo totale: {(time.time()-t0)/60:.1f} min | best val IoU: {best_iou:.4f}")


## Valutazione finale sul test set

In [ ]:
model.load_state_dict(torch.load(os.path.join(RUN_DIR, "best_unet.pt"), map_location=DEVICE))
test_loss, test_iou = evaluate(model, dl_te)
prec = f"test loss {test_loss:.4f} | test IoU {test_iou:.4f}"
print(prec)
with open(os.path.join(RUN_DIR, "results.txt"), "w") as f:
    f.write(prec + "\n")
# Prossimo passo (stadio 2, §7.3 handoff): dalla mappa di probabilita` al catalogo
# threshold -> scipy.ndimage.label -> centroidi, raggio equivalente, massa integrata (calibrazione mpu)
# -> completeness/purity vs groups_090.hdf5, poi HMF.
